In [ ]:
import torch
import torchcodec
from pathlib import Path
from datasets import Dataset, DatasetDict, Audio
import re

01. Environment Setup
02. Explore AISHELL Dataset
03. Match Audio and Transcriptions
04. Build Dataset with Generators
05. Create Hanzi and Pinyin Datasets
06. Data Cleaning and Validation
07. Build CTC Vocabularies
08. Initialize Wav2Vec2 Processors
09. Prepare Audio and Labels
10. Fine-tune Hanzi ASR
11. Fine-tune Pinyin ASR
12. Evaluation and Comparison

# Environmental Setup and Get Files

In [10]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "processed_data").exists():
    candidate = (
        PROJECT_ROOT
        / "semester-project-chinese-ASR"
    )

    if (candidate / "processed_data").exists():
        PROJECT_ROOT = candidate
    else:
        raise FileNotFoundError(
            "Cannot find processed_data directory."
        )

DATA_ROOT = PROJECT_ROOT / "aishell3"
PROCESSED_DATA_DIR = PROJECT_ROOT / "processed_data"

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_DATA_DIR)

Project root: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR
Processed data: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/processed_data


In [11]:
# already have dataset, laod the dataset, get the miniset
import random
from datasets import load_from_disk, DatasetDict

full_hanzi_dataset = load_from_disk(
    PROCESSED_DATA_DIR / "aishell3_hanzi"
)

full_pinyin_dataset = load_from_disk(
    PROCESSED_DATA_DIR / "aishell3_pinyin"
)

print("Full Hanzi:", full_hanzi_dataset)
print("Full Pinyin:", full_pinyin_dataset)

Loading dataset from disk:   0%|          | 0/36 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/36 [00:00<?, ?it/s]

Full Hanzi: DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 56935
    })
    validation: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 6327
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 24773
    })
})
Full Pinyin: DatasetDict({
    train: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 56935
    })
    validation: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 6327
    })
    test: Dataset({
        features: ['id', 'speaker_id', 'audio', 'text', 'age_group', 'gender', 'accent'],
        num_rows: 24773
    })
})


In [12]:
splits = ["train", "validation", "test"]

for split in splits:
    assert (
        full_hanzi_dataset[split]["id"]
        == full_pinyin_dataset[split]["id"]
    )

print("Full datasets are aligned.")

Full datasets are aligned.


In [13]:
SUBSET_FRACTION = 0.1

split_seeds = {
    "train": 42,
    "validation": 43,
    "test": 44,
}

subset_indices = {}

for split in splits:
    dataset_size = len(
        full_hanzi_dataset[split]
    )

    subset_size = max(
        1,
        round(dataset_size * SUBSET_FRACTION),
    )

    rng = random.Random(split_seeds[split])

    indices = rng.sample(
        range(dataset_size),
        k=subset_size,
    )

    subset_indices[split] = sorted(indices)

In [14]:
hanzi_dataset = DatasetDict({
    split: full_hanzi_dataset[split].select(
        subset_indices[split]
    )
    for split in splits
})

pinyin_dataset = DatasetDict({
    split: full_pinyin_dataset[split].select(
        subset_indices[split]
    )
    for split in splits
})

In [15]:
for split in splits:
    assert (
        hanzi_dataset[split]["id"]
        == pinyin_dataset[split]["id"]
    )

    print(
        f"{split}: "
        f"{len(full_hanzi_dataset[split])} "
        f"-> {len(hanzi_dataset[split])}"
    )

print("10% subsets are aligned.")

train: 56935 -> 5694
validation: 6327 -> 633
test: 24773 -> 2477
10% subsets are aligned.


In [16]:
del full_hanzi_dataset
del full_pinyin_dataset

# Start to build CTC vocab. Must have blank, unk, and | to separete repeated phones. 
1. Hanzi: each token contains every character
2. pinyin: each token contains pinyin+tone
3. phone: in alphabet a to z

In [17]:
hanzi_characters = sorted({
    character
    for text in hanzi_dataset["train"]["text"]
    for character in text
}) # getting the unique characters in the training set

hanzi_vocab = {
    character: index
    for index, character in enumerate(hanzi_characters)
} # vocab to id

# adding special tokens for unknown and padding
hanzi_vocab["[UNK]"] = len(hanzi_vocab)
hanzi_vocab["[PAD]"] = len(hanzi_vocab)
hanzi_vocab["|"] = len(hanzi_vocab)

print("Hanzi vocabulary size:", len(hanzi_vocab))
print(list(hanzi_vocab.items())[:20])

Hanzi vocabulary size: 2697
[('一', 0), ('丁', 1), ('七', 2), ('万', 3), ('丈', 4), ('三', 5), ('上', 6), ('下', 7), ('不', 8), ('与', 9), ('丐', 10), ('丑', 11), ('专', 12), ('且', 13), ('丕', 14), ('世', 15), ('丘', 16), ('丙', 17), ('业', 18), ('东', 19)]


In [18]:
def read_train_pinyin_tokens(phone_set_path):
    pinyin_tokens = []
    reading_train_section = False

    with Path(phone_set_path).open(
        "r",
        encoding="utf-8"
    ) as f:
        for line in f:
            line = line.strip()

            if line == "# The phones frequencies of the train set.":
                reading_train_section = True
                continue

            if line == "# The phones frequencies of the test set.":
                break

            if not reading_train_section:
                continue

            if not line or line.startswith("#"):
                continue

            pinyin_token, frequency = line.split()

            if not re.fullmatch(r"[a-z]+[1-5]", pinyin_token):
                raise ValueError(
                    f"Invalid Pinyin token: {pinyin_token}"
                )

            pinyin_tokens.append(pinyin_token)

    if len(pinyin_tokens) != len(set(pinyin_tokens)):
        raise ValueError(
            "Duplicate Pinyin tokens found in training section."
        )

    return sorted(pinyin_tokens)

In [19]:
pinyin_tokens = read_train_pinyin_tokens(
    DATA_ROOT / "phone_set.txt"
)

pinyin_vocab = {
    token: index
    for index, token in enumerate(pinyin_tokens)
}

pinyin_vocab["|"] = len(pinyin_vocab)
pinyin_vocab["[UNK]"] = len(pinyin_vocab)
pinyin_vocab["[PAD]"] = len(pinyin_vocab)

print("Pinyin vocabulary size:", len(pinyin_vocab))
print(list(pinyin_vocab.items())[:20])

Pinyin vocabulary size: 1847
[('a1', 0), ('a2', 1), ('a3', 2), ('a4', 3), ('a5', 4), ('ai1', 5), ('ai2', 6), ('ai3', 7), ('ai4', 8), ('ai5', 9), ('an1', 10), ('an2', 11), ('an3', 12), ('an4', 13), ('ang1', 14), ('ang2', 15), ('ang4', 16), ('ao1', 17), ('ao2', 18), ('ao3', 19)]


In [20]:
phone_tokens = list("abcdefghijklmnopqrstuvwxyz12345")

phone_vocab = {
    token: index
    for index, token in enumerate(phone_tokens)
}

phone_vocab["|"] = len(phone_vocab)
phone_vocab["[UNK]"] = len(phone_vocab)
phone_vocab["[PAD]"] = len(phone_vocab)

print("Phone vocabulary size:", len(phone_vocab))
print(phone_vocab)

Phone vocabulary size: 34
{'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6, 'h': 7, 'i': 8, 'j': 9, 'k': 10, 'l': 11, 'm': 12, 'n': 13, 'o': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25, '1': 26, '2': 27, '3': 28, '4': 29, '5': 30, '|': 31, '[UNK]': 32, '[PAD]': 33}


Question: How does CTC handle Chinese characters that have multiple pronunciations (polyphonic characters)?
CTC + n-gram: how to sampling

In [21]:
import json
from pathlib import Path

vocab_names = ["hanzi", "pinyin", "phone"]

vocab_list = [hanzi_vocab, pinyin_vocab, phone_vocab]

VOCAB_DIR = Path("vocab")
VOCAB_DIR.mkdir(parents=True, exist_ok=True)

for vocab_name, vocabulary in zip(vocab_names, vocab_list):
    vocab_path = VOCAB_DIR / f"{vocab_name}_vocab.json"

    with vocab_path.open("w", encoding="utf-8") as vocab_file:
        json.dump(
            vocabulary,
            vocab_file,
            ensure_ascii=False,
            indent=2
        )

    print(
        f"Saved {vocab_name} vocabulary: "
        f"{vocab_path} "
        f"({len(vocabulary)} tokens)"
    )

Saved hanzi vocabulary: vocab/hanzi_vocab.json (2697 tokens)
Saved pinyin vocabulary: vocab/pinyin_vocab.json (1847 tokens)
Saved phone vocabulary: vocab/phone_vocab.json (34 tokens)


# CTC tokenizer

In [22]:
from transformers import Wav2Vec2CTCTokenizer

hanzi_vocab_path = VOCAB_DIR / "hanzi_vocab.json"
assert hanzi_vocab_path.exists(), (
    f"Vocabulary not found: {hanzi_vocab_path.resolve()}"
)

phone_vocab_path = VOCAB_DIR / "phone_vocab.json"
assert phone_vocab_path.exists(), (
    f"Vocabulary not found: {phone_vocab_path.resolve()}"
)

hanzi_tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(hanzi_vocab_path),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

phone_tokenizer = Wav2Vec2CTCTokenizer(
    vocab_file=str(phone_vocab_path),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

print("Tokenizer size:", len(hanzi_tokenizer))
print("Vocab file:", hanzi_vocab_path.resolve())
print("Phone Tokenizer size:", len(phone_tokenizer))
print("Phone Vocab file:", phone_vocab_path.resolve())

Tokenizer size: 2699
Vocab file: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/vocab/hanzi_vocab.json
Phone Tokenizer size: 36
Phone Vocab file: /Users/shuangping/Desktop/ASR_Project/semester-project-chinese-ASR/vocab/phone_vocab.json


In [23]:
class PinyinSyllableCTCTokenizer(Wav2Vec2CTCTokenizer):
    def _tokenize(self, text, **kwargs):
        syllables = text.lower().split()
        tokens = []

        for index, syllable in enumerate(syllables):
            if index > 0:
                tokens.append(self.word_delimiter_token)

            tokens.append(syllable)

        return tokens

pinyin_tokenizer = PinyinSyllableCTCTokenizer(
    vocab_file=str(VOCAB_DIR / "pinyin_vocab.json"),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
)

In [24]:
hanzi_example = hanzi_dataset["train"][0]["text"]

hanzi_encoding = hanzi_tokenizer(hanzi_example)

print("Text:", hanzi_example)
print("IDs:", hanzi_encoding.input_ids[:20])
print(
    "Tokens:",
    hanzi_tokenizer.convert_ids_to_tokens(
        hanzi_encoding.input_ids[:20]
    ),
)

Text: 奇迹动物园
IDs: [582, 2383, 306, 1583, 493]
Tokens: ['奇', '迹', '动', '物', '园']


In [25]:
# check the decoding of the tokenized output
pinyin_example_1 = "er2 zi5 wanr2 shou3 ji1"
pinyin_example_2 = "er2zi5wanr2shou3ji1"

encoding_1 = pinyin_tokenizer(pinyin_example_1)
encoding_2 = pinyin_tokenizer(pinyin_example_2)

tokens_1 = pinyin_tokenizer.convert_ids_to_tokens(
    encoding_1.input_ids
)
tokens_2 = pinyin_tokenizer.convert_ids_to_tokens(
    encoding_2.input_ids
)

decoded_1 = pinyin_tokenizer.decode(
    encoding_1.input_ids,
    group_tokens=False,
)

decoded_2 = pinyin_tokenizer.decode(
    encoding_2.input_ids,
    group_tokens=False,
)

print("Input:", pinyin_example_1)
print("Tokens:", tokens_1)
print("Decoded:", decoded_1)

print("Input:", pinyin_example_2)
print("Tokens:", tokens_2)
print("Decoded:", decoded_2)

Input: er2 zi5 wanr2 shou3 ji1
Tokens: ['er2', 'zi5', 'wanr2', 'shou3', 'ji1']
Decoded: er2zi5wanr2shou3ji1
Input: er2zi5wanr2shou3ji1
Tokens: ['er2', 'zi5', 'wanr2', 'shou3', 'ji1']
Decoded: er2zi5wanr2shou3ji1


# Processor, combined audio and text together

In [27]:
from transformers import Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16_000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

from transformers import Wav2Vec2Processor

hanzi_processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=hanzi_tokenizer,
)

pinyin_processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=pinyin_tokenizer,
)

# phone_processor = Wav2Vec2Processor(
#     feature_extractor=feature_extractor,
#     tokenizer=phone_tokenizer,
# )

In [28]:
def prepare_dataset(example, processor):
    audio = example["audio"]

    processed = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],
        text=example["text"],
    )

    example["input_values"] = (
        processed["input_values"][0]
    )

    example["input_length"] = len(
        example["input_values"]
    )

    example["labels"] = processed["labels"]

    return example

#{
#     "id": "SSB00050001",
#     "audio": {...},
#     "text": "广州女大学生……",
#     "input_values": [...],
#     "input_length": 123456,
#     "labels": [...]
# }

In [29]:
prepared_hanzi_dataset = hanzi_dataset.map(
    prepare_dataset,
    fn_kwargs={
        "processor": hanzi_processor,
    }, desc="Preparing Hanzi dataset",)

prepared_pinyin_dataset = pinyin_dataset.map(
    prepare_dataset,
    fn_kwargs={
        "processor": pinyin_processor,
    },
    desc="Preparing Pinyin dataset"
)

# prepared_phone_dataset = phone_dataset.map(prepare_dataset, fn_kwargs = {"processor": phone_processor}, desc = "Preparing Phone Dataset")

Preparing Hanzi dataset:   0%|          | 0/5694 [00:00<?, ? examples/s]

Preparing Hanzi dataset:   0%|          | 0/633 [00:00<?, ? examples/s]

Preparing Hanzi dataset:   0%|          | 0/2477 [00:00<?, ? examples/s]

Preparing Pinyin dataset:   0%|          | 0/5694 [00:00<?, ? examples/s]

Preparing Pinyin dataset:   0%|          | 0/633 [00:00<?, ? examples/s]

Preparing Pinyin dataset:   0%|          | 0/2477 [00:00<?, ? examples/s]

In [32]:
test_example = prepare_dataset(
    hanzi_dataset["train"][0],
    hanzi_processor,
)

print("Input length:", test_example["input_length"])
print("Number of labels:", len(test_example["labels"]))
print("true labels:", test_example["text"])
print("First input values:",
      test_example["input_values"][:10])
print("Labels:", test_example["labels"])

Input length: 31424
Number of labels: 5
true labels: 奇迹动物园
First input values: [0.00024634 0.00024634 0.00024634 0.00024634 0.00024634 0.00024634
 0.00024634 0.00024634 0.00024634 0.00024634]
Labels: [582, 2383, 306, 1583, 493]


# data colattor


In [33]:
import torch

from dataclasses import dataclass
from typing import Any
from transformers import Wav2Vec2Processor


@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: bool | str = True

    def __call__(
        self,
        features: list[dict[str, Any]],
    ) -> dict[str, torch.Tensor]:

        # Audio inputs: floating-point waveforms
        input_features = [
            {
                "input_values": feature[
                    "input_values"
                ]
            }
            for feature in features
        ]

        # Text labels: integer token IDs
        label_features = [
            {
                "input_ids": feature["labels"]
            }
            for feature in features
        ]

        # Pad audio using the feature extractor
        batch = (
            self.processor.feature_extractor.pad(
                input_features,
                padding=self.padding,
                return_tensors="pt",
            )
        )

        # Pad labels using the tokenizer
        labels_batch = (
            self.processor.tokenizer.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )
        )

        # Ignore label padding when computing CTC loss
        labels = labels_batch[
            "input_ids"
        ].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100,
        )

        batch["labels"] = labels

        return batch

In [34]:
hanzi_data_collator = (
    DataCollatorCTCWithPadding(
        processor=hanzi_processor,
        padding=True,
    )
)

pinyin_data_collator = (
    DataCollatorCTCWithPadding(
        processor=pinyin_processor,
        padding=True,
    )
)

In [35]:
hanzi_features = [
    prepared_hanzi_dataset["train"][0],
    prepared_hanzi_dataset["train"][1],
]

hanzi_batch = hanzi_data_collator(
    hanzi_features
)

print(
    "input_values:",
    hanzi_batch["input_values"].shape,
)

print(
    "attention_mask:",
    hanzi_batch["attention_mask"].shape,
)

print(
    "labels:",
    hanzi_batch["labels"].shape,
)

print(hanzi_batch["labels"])

input_values: torch.Size([2, 38078])
attention_mask: torch.Size([2, 38078])
labels: torch.Size([2, 8])
tensor([[ 582, 2383,  306, 1583,  493, -100, -100, -100],
        [ 503,  687,  665, 1980,  118,  230, 1683,  597]])
